# Dog eye screening: baseline

Runs the whole baseline on a free Colab GPU: download the data, extract DINOv2 features, fold near-duplicates into groups, split by group, train a logistic-regression baseline and evaluate it.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. Get a free Roboflow API key at https://app.roboflow.com/settings/api and add it as a Colab secret named `ROBOFLOW_API_KEY` (key icon in the left sidebar). Your key stays in your Colab; it is never written to the repo.

In [ ]:
REPO_URL = "https://github.com/zhengjingtian5-dotcom/dog-eye-screening.git"

!git clone -q $REPO_URL dog-eye-screening
%cd dog-eye-screening
!pip install -q -e ".[data,features]"

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["ROBOFLOW_API_KEY"] = getpass("Roboflow API key: ")

## 1. Download and crop the images

In [ ]:
!python -m dogeye.data --out data

## 2. Extract DINOv2 features (about 5 minutes on a T4)

In [ ]:
!python -m dogeye.features --manifest data/manifest.csv --out data/features.npz

## 3. Group near-duplicates and split by group

Copies of the same photo must not appear in both train and test, or the scores are inflated.

In [ ]:
!python -m dogeye.groups --manifest data/manifest.csv --features data/features.npz --out data/manifest_split.csv

## 4. Train and evaluate

The threshold is set on the validation split to catch at least 95% of eyes that need a vet.

In [ ]:
!python -m dogeye.train --manifest data/manifest_split.csv --features data/features.npz --out runs/baseline

In [ ]:
import json
from IPython.display import Image, display
import pandas as pd

print(json.dumps(json.load(open("runs/baseline/metrics.json"))["test"], indent=2))
display(pd.read_csv("runs/baseline/per_condition.csv"))
display(Image("runs/baseline/confusion_matrix.png"))
display(Image("runs/baseline/reliability.png"))

## 5. Look at the misses

The most useful next step is to look at the see_vet eyes the model missed.

In [ ]:
preds = pd.read_csv("runs/baseline/test_predictions.csv")
misses = preds[(preds.label == "see_vet") & (preds.flagged == 0)].sort_values("prob_see_vet")
print(len(misses), "missed")
for path in misses.path.head(8):
    display(Image(path, width=160))

## 6. Keep the results

Download `baseline_results.zip` (model, metrics, plots, predictions). The model file `runs/baseline/model.joblib` is small and is what the web demo loads.

In [ ]:
!zip -qr baseline_results.zip runs/baseline
from google.colab import files
files.download("baseline_results.zip")